In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('train-test.csv')
print(df.head())

     load_id        pickup      delivery  pickup_lat  pickup_lon  \
0  TR-000001      Richmond     Baltimore    38.09122   -76.78906   
1  TR-000002      Richmond  Philadelphia    38.09122   -76.78906   
2  TR-000003  Philadelphia     Green Bay    39.22317   -72.96710   
3  TR-000004      Hartford       Atlanta    39.55328   -72.18051   
4  TR-000005        Dallas     Nashville    31.83025   -94.38343   

   delivery_lat  delivery_lon  distance equipment   weight        date  \
0      38.16908     -72.74564     274.3   Dry Van  30658.0  2025-01-01   
1      39.22317     -72.96710     280.5    Reefer  17555.0  2025-01-01   
2      44.30296     -87.52871     967.8   Dry Van  31721.0  2025-01-01   
3      34.84933     -86.28940     965.4   Dry Van  32333.0  2025-01-01   
4      35.29479     -88.08915     541.9    Reefer  35183.0  2025-01-01   

   market_index  quote_signal  posted_rate  
0       0.95684       2.39595       645.41  
1       0.97623       2.43355       679.97  
2       1.0

In [3]:
print(df.shape)
print(df.columns)
print(df.dtypes)

(48000, 14)
Index(['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
       'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight',
       'date', 'market_index', 'quote_signal', 'posted_rate'],
      dtype='object')
load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object


In [4]:
print(df.isnull().sum())

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64


In [5]:
print("duplicate rows:", df.duplicated().sum())
print("duplicate loads:",df["load_id"].duplicated().sum())

duplicate rows: 0
duplicate loads: 0


In [6]:
df["date"] = pd.to_datetime(df["date"])

In [7]:
print("Earliest date:", df["date"].min())
print("Latest date:", df["date"].max())
print("Unique dates:", df["date"].nunique())

Earliest date: 2025-01-01 00:00:00
Latest date: 2025-10-31 00:00:00
Unique dates: 304


## Assessment Notes
Working Notes — Stage 1: Initial Data Inspection
Dataset: train-test.csv

Shape:
48,000 rows
14 columns
Columns:
load_id
pickup
delivery
pickup_lat
pickup_lon
delivery_lat
delivery_lon
distance
equipment
weight
date
market_index
quote_signal
posted_rate
Data types

Categorical / text-like:
load_id
pickup
delivery
equipment
date was initially loaded as an object/string and then converted to datetime
Remaining columns are numerical float64

Missing values:
weight: 300 missing values
market_index: 374 missing values
All other columns: no missing values

Duplicate checks:
Duplicate rows: 0
Duplicate load_id values: 0

Date coverage:
Earliest date: 2025-01-01
Latest date: 2025-10-31
Unique dates: 304

Initial interpretation:
This is a supervised tabular regression problem
Target: posted_rate
posted_rate appears to represent the total freight rate for a load, not dollars per mile
Validation consideration
Because the dataset spans multiple months in chronological order, a time-based train/validation split may be more realistic than a random split
This would better simulate the real situation of training on past data and predicting future freight rates
Early observation


In [8]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,48000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,2025-05-31 19:53:29.400000,1.083387,2.062468,2373.980682
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,2025-03-18 00:00:00,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,2025-05-31 00:00:00,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,2025-08-15 00:00:00,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,NaN,0.168091,0.291391,1486.493245


## weight investigation

In [9]:
print("Negative weights:", (df["weight"] < 0).sum())

Negative weights: 292


In [10]:
df[df["weight"] < 0].head(20)

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
68,TR-000069,Amarillo,Dayton,33.17137,-104.36789,39.87206,-85.62931,1334.2,Reefer,-36559.0,2025-01-01,0.96102,2.11777,2831.27
204,TR-000205,Fresno,Tucson,33.07921,-119.79528,30.92733,-113.23037,496.2,Dry Van,-26670.0,2025-01-02,0.96903,2.13155,1046.92
206,TR-000207,Columbia,Raleigh,34.63570,-83.28506,35.37376,-77.47605,411.3,Reefer,-20003.0,2025-01-02,0.94800,2.31682,959.15
225,TR-000226,Greensboro,Lubbock,35.98195,-82.78379,29.87534,-99.85456,1317.4,Reefer,-23981.0,2025-01-02,1.00820,2.13918,2836.00
376,TR-000377,Madison,Providence,41.70243,-89.85588,37.37287,-69.50000,1331.1,Dry Van,-41397.0,2025-01-03,0.94787,1.91867,2522.54
446,TR-000447,Albuquerque,Albany,35.39918,-109.59792,43.96975,-71.05842,2438.3,Reefer,-37215.0,2025-01-03,0.96575,1.96959,4841.57
495,TR-000496,Montgomery,Dayton,31.37602,-86.10743,39.87206,-85.62931,715.9,Dry Van,-31214.0,2025-01-04,0.85153,2.03947,1473.78
641,TR-000642,Tulsa,Indianapolis,33.45894,-94.78556,38.52567,-88.85948,582.4,Dry Van,-41023.0,2025-01-05,0.81988,2.22721,1264.33
806,TR-000807,Cincinnati,Buffalo,37.15706,-86.77499,41.67681,-78.56704,676.1,Dry Van,-28320.0,2025-01-06,0.79158,1.95680,1307.76
1093,TR-001094,Cincinnati,Albany,37.15706,-86.77499,43.96975,-71.05842,1058.5,Dry Van,-25153.0,2025-01-07,0.89891,1.81081,1907.83


Negative weight values were identified. Inspection showed that their absolute magnitudes were consistent with normal weight ranges for the corresponding equipment types, suggesting likely sign corruption rather than arbitrary invalid values.

In [11]:
print("Negative weights:", (df["weight"] < 0).sum())
print("Zero weights:", (df["weight"] == 0).sum())
print("Positive weights:", (df["weight"] > 0).sum())

Negative weights: 292
Zero weights: 0
Positive weights: 47408


In [12]:
negative_abs = df.loc[df["weight"] < 0, "weight"].abs()
positive = df.loc[df["weight"] > 0, "weight"]

print("Negative abs mean:", negative_abs.mean())
print("Negative abs median:", negative_abs.median())

print("Positive mean:", positive.mean())
print("Positive median:", positive.median())

Negative abs mean: 31724.195205479453
Negative abs median: 31821.5
Positive mean: 31415.35867364158
Positive median: 31493.5


In [13]:
missing_weight_rows = df.loc[
    df["weight"].isnull(),
    [
        "load_id",
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "date",
        "market_index",
        "quote_signal",
        "posted_rate"
    ]
]

missing_weight_rows.head(20)

,load_id,pickup,delivery,distance,equipment,date,market_index,quote_signal,posted_rate
323,TR-000324,Richmond,San Francisco,2848.8,Reefer,2025-01-03,0.91370,1.92679,5430.07
648,TR-000649,Tulsa,Mobile,467.0,Reefer,2025-01-05,0.77536,2.48162,1178.03
694,TR-000695,Los Angeles,Fresno,432.6,Reefer,2025-01-05,0.84069,2.37036,1022.13
732,TR-000733,El Paso,Hartford,2497.6,Reefer,2025-01-05,0.79026,2.01004,4991.60
911,TR-000912,Washington,Kansas City,887.8,Dry Van,2025-01-06,0.81120,1.85853,1643.25
942,TR-000943,Jacksonville,Corpus Christi,833.9,Dry Van,2025-01-06,0.79617,1.90050,1602.03
963,TR-000964,Greensboro,Madison,640.8,Reefer,2025-01-07,0.89852,2.30231,1466.95
1009,TR-001010,Richmond,Savannah,550.2,Dry Van,2025-01-07,0.89645,2.10347,1161.37
1021,TR-001022,Albuquerque,Reno,678.8,Dry Van,2025-01-07,0.86853,1.91635,1318.04
1058,TR-001059,Charleston,Indianapolis,714.8,Dry Van,2025-01-07,0.85299,1.97105,1431.30


In [14]:
print(missing_weight_rows["equipment"].value_counts())
print(df["equipment"].value_counts())

equipment
Dry Van    175
Reefer      70
Flatbed     55
Name: count, dtype: int64
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64


In [15]:
missing_rate_by_equipment = df.groupby("equipment")["weight"].apply(lambda x: x.isnull().mean() * 100)

print(missing_rate_by_equipment)

equipment
Dry Van    0.643335
Flatbed    0.628356
Reefer     0.581154
Name: weight, dtype: float64


In [16]:
df.assign(weight_abs=df["weight"].abs()).groupby("equipment")["weight_abs"].agg(
    ["mean", "median", "count"]
)

,mean,median,count
equipment,,,
Dry Van,31389.812928,31444.0,27027
Flatbed,31480.436307,31532.5,8698
Reefer,31433.275992,31577.0,11975


In [17]:
df["weight"] = df["weight"].abs()

df["weight"] = df["weight"].fillna(df["weight"].median())

In [18]:
print("Negative weights:", (df["weight"] < 0).sum())
print("Missing weights:", df["weight"].isnull().sum())

Negative weights: 0
Missing weights: 0


## Working Notes — Weight Cleaning
Issue identified:
weight contained 300 missing values
weight also contained 292 negative values
Negative freight weight is physically invalid

Investigation:
The absolute values of negative weights had a similar distribution to valid positive weights:
negative absolute mean ≈ 31,724
negative absolute median ≈ 31,822
positive mean ≈ 31,415
positive median ≈ 31,494
Missing-weight percentages were also very similar across equipment types:
Dry Van ≈ 0.64%
Flatbed ≈ 0.63%
Reefer ≈ 0.58%
Median weights across equipment types were very close, around 31.4k–31.6k lb

Decision:
I treated negative values as likely sign errors and corrected them using absolute value
I also filled missing weights using the global median, since equipment specific medians were nearly identical

## Market index investigation 

In [19]:
print (df["market_index"].describe())

count    47626.000000
mean         1.083387
std          0.168091
min          0.676390
25%          0.949670
50%          1.055800
75%          1.219590
max          1.467780
Name: market_index, dtype: float64


In [20]:
print("Missing market_index:", df["market_index"].isnull().sum())

missing_market_rows = df.loc[
    df["market_index"].isnull(),
    ["date", "pickup", "delivery", "distance", "equipment", "quote_signal", "posted_rate"]
]

missing_market_rows.head(20)

Missing market_index: 374


,date,pickup,delivery,distance,equipment,quote_signal,posted_rate
116,2025-01-01,Memphis,Fort Wayne,533.8,Reefer,2.37177,1271.10
226,2025-01-02,Tampa,Hartford,1342.7,Reefer,2.01837,2743.13
311,2025-01-02,Los Angeles,New York,2935.7,Flatbed,1.91236,5568.57
511,2025-01-04,Grand Rapids,Amarillo,1420.7,Reefer,1.94596,2786.63
531,2025-01-04,Richmond,San Antonio,1624.7,Reefer,2.04970,3295.80
629,2025-01-05,Birmingham,Fort Wayne,648.1,Dry Van,1.93237,1256.20
644,2025-01-05,Lubbock,Milwaukee,1089.3,Dry Van,2.04466,2171.37
905,2025-01-06,Fort Wayne,Toledo,80.0,Reefer,2.92545,231.54
1012,2025-01-07,Chattanooga,Lubbock,978.6,Reefer,2.31501,2236.76
1274,2025-01-08,Little Rock,Jacksonville,573.4,Dry Van,2.06009,1186.43


In [21]:
df.groupby(df["date"].dt.month)["market_index"].agg(["mean", "median", "count"])

,mean,median,count
date,,,
1,0.926696,0.932905,4882
2,0.995997,0.993130,4301
3,1.072880,1.064850,4992
4,1.207649,1.207125,4788
5,1.301452,1.304025,4870
6,1.280316,1.275130,4741
7,1.200893,1.212100,4876
8,0.981480,0.982340,4726
9,0.893039,0.887810,4639


In [22]:
daily_market_stats = df.groupby("date")["market_index"].agg(["mean", "median", "std", "count"])

daily_market_stats.head(10)

,mean,median,std,count
date,,,,
2025-01-01,0.961224,0.962590,0.024493,144
2025-01-02,0.986606,0.989030,0.025281,166
2025-01-03,0.940339,0.941040,0.026063,156
2025-01-04,0.857702,0.855715,0.023678,154
2025-01-05,0.801855,0.800030,0.024737,152
2025-01-06,0.813340,0.813120,0.027817,167
2025-01-07,0.888123,0.888890,0.025898,160
2025-01-08,0.973030,0.971425,0.023872,188
2025-01-09,1.000927,0.998300,0.023554,146


In [23]:
daily_market_stats.tail(10)

,mean,median,std,count
date,,,,
2025-10-22,1.016604,1.015505,0.023585,146
2025-10-23,1.047030,1.047950,0.026197,155
2025-10-24,0.995974,0.992970,0.024142,125
2025-10-25,0.910241,0.906950,0.026845,163
2025-10-26,0.847234,0.847370,0.024634,145
2025-10-27,0.857300,0.858215,0.024855,146
2025-10-28,0.926082,0.923350,0.022158,131
2025-10-29,1.008277,1.010020,0.024045,162
2025-10-30,1.035295,1.035260,0.022253,177


In [24]:
df["market_index"] = df["market_index"].fillna(df.groupby("date")["market_index"].transform("median"))

In [25]:
print("Missing market_index:", df["market_index"].isnull().sum())

Missing market_index: 0


## Market Index Investigation
- 374 missing market_index values were identified.
- Monthly statistics showed substantial temporal variation in the feature.
- Daily analysis showed approximately 125–188 valid observations per date.
- Within-day standard deviation was consistently low (~0.02–0.03), while daily means changed noticeably over time.
- This suggests market_index represents a date dependent market condition.
Decision
- Missing values will be imputed using the median market_index from the same date, rather than the global median.
## In conclusion 
Rather than using a global median as i did for weight missing values , i discovered that market index depend heavily in date so it needs a different approach  

In [26]:
numeric_cols = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "posted_rate"
]

correlation_matrix = df[numeric_cols].corr()

correlation_matrix["posted_rate"].sort_values(ascending=False)

posted_rate     1.000000
distance        0.908519
weight          0.041229
market_index    0.034172
quote_signal   -0.039858
pickup_lat     -0.090873
delivery_lat   -0.091970
pickup_lon     -0.255058
delivery_lon   -0.257086
Name: posted_rate, dtype: float64

In [27]:
df["rate_per_mile"] = df["posted_rate"] / df["distance"]

In [28]:
print(df[["quote_signal", "rate_per_mile"]].corr())

               quote_signal  rate_per_mile
quote_signal       1.000000       0.049342
rate_per_mile      0.049342       1.000000


In [29]:
df["posted_rate"].describe()

count    48000.000000
mean      2373.980682
std       1486.493245
min         57.220000
25%       1251.555000
50%       2030.760000
75%       3330.750000
max      25533.000000
Name: posted_rate, dtype: float64

In [30]:
print("95th percentile:", df["posted_rate"].quantile(0.95))
print("99th percentile:", df["posted_rate"].quantile(0.99))
print("99.5th percentile:", df["posted_rate"].quantile(0.995))

95th percentile: 4953.7665
99th percentile: 5972.834000000003
99.5th percentile: 6510.9689999999955


In [31]:
df.nlargest(
    20,
    "posted_rate"
)[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "market_index",
        "quote_signal",
        "posted_rate"
    ]
]

,pickup,delivery,distance,equipment,weight,date,market_index,quote_signal,posted_rate
12184,Bakersfield,Hartford,2829.8,Dry Van,33153.0,2025-03-19,1.16556,1.83188,25533.00
37764,Phoenix,Syracuse,2810.9,Dry Van,33674.0,2025-08-27,1.04877,2.16284,24294.98
1466,Los Angeles,Syracuse,2786.0,Reefer,30322.0,2025-01-10,0.98678,1.85370,24140.21
17371,Reno,Philadelphia,2777.6,Reefer,33158.0,2025-04-20,1.15070,2.15601,23662.71
26235,Charleston,Phoenix,2552.5,Dry Van,25665.0,2025-06-15,1.20224,1.81289,23580.42
28219,Tucson,Raleigh,2423.3,Dry Van,30972.0,2025-06-27,1.32894,1.97112,22755.66
3353,Phoenix,Charleston,2527.9,Reefer,36840.0,2025-01-22,1.03241,2.07342,22534.65
40171,Montgomery,Fresno,2283.4,Dry Van,32447.0,2025-09-11,0.98502,1.93645,20361.89
40096,Baltimore,Oklahoma City,1760.3,Reefer,29521.0,2025-09-11,1.00464,2.28753,20132.37
47298,Boston,Bakersfield,2979.1,Reefer,30585.0,2025-10-27,0.86501,2.09472,19110.30


In [32]:
df.groupby("equipment")["posted_rate"].agg(["mean", "median", "std", "count"])

,mean,median,std,count
equipment,,,,
Dry Van,2271.548686,1953.035,1423.029914,27202
Flatbed,2445.087223,2076.810,1505.053087,8753
Reefer,2553.636939,2196.670,1589.670824,12045


In [33]:
df.groupby("equipment")["rate_per_mile"].agg(["mean", "median", "std", "count"])

,mean,median,std,count
equipment,,,,
Dry Van,2.115401,2.046203,0.538102,27202
Flatbed,2.294729,2.219683,0.610187,8753
Reefer,2.383376,2.311540,0.615467,12045


Equipment type showed a consistent pricing effect. Reefer loads had the highest mean and median posted rates and rate-per-mile, followed by Flatbed and Dry Van, suggesting equipment type is an informative categorical feature.

In [34]:
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()

In [35]:
monthly_rate_stats = df.groupby("month")["posted_rate"].agg(["mean", "median", "std", "count"])

monthly_rate_stats

,mean,median,std,count
month,,,,
1,2255.967048,1915.200,1454.274952,4918
2,2273.804801,1994.250,1317.077105,4337
3,2372.268092,2022.920,1462.531615,5036
4,2372.162308,2044.240,1463.102457,4819
5,2421.776342,2065.510,1486.071436,4913
6,2497.030115,2120.220,1606.707302,4783
7,2415.162030,2059.145,1504.943296,4912
8,2338.407661,2015.730,1474.979672,4759
9,2406.374013,2057.130,1523.483983,4670


In [36]:
monthly_market_stats = df.groupby("month")["market_index"].agg(["mean", "median"])

monthly_market_stats

,mean,median
month,,
1,0.926590,0.932655
2,0.996058,0.993240
3,1.072974,1.064850
4,1.207442,1.206830
5,1.301281,1.303100
6,1.280098,1.274800
7,1.200762,1.211925
8,0.981595,0.982540
9,0.893135,0.887825


In [37]:
weekday_rate_stats = df.groupby("day_of_week")["posted_rate"].agg(["mean", "median", "count"])

weekday_rate_stats

,mean,median,count
day_of_week,,,
Friday,2376.849855,2029.100,6975
Monday,2349.308676,2015.110,6769
Saturday,2373.714334,2044.845,6770
Sunday,2335.073176,2022.035,6808
Thursday,2403.265599,2037.650,6934
Tuesday,2372.890626,2029.630,6737
Wednesday,2405.086706,2036.580,7007


In [38]:
weekday_market_stats = df.groupby("day_of_week")["market_index"].agg(["mean", "median"])
weekday_market_stats

,mean,median
day_of_week,,
Friday,1.134331,1.08615
Monday,0.999806,0.95978
Saturday,1.047617,1.00472
Sunday,0.988635,0.94380
Thursday,1.178609,1.13454
Tuesday,1.075413,1.03489
Wednesday,1.153319,1.10845


## Working notes — Time effects
- posted_rate showed moderate monthly variation, with the highest average and median occurring around June.
- market_index showed considerably stronger temporal variation across months.
- The movements in posted_rate did not directly mirror the magnitude of changes in market_index, suggesting that freight price depends on several interacting features.
- Weekday differences in posted_rate were relatively small.
- month appears potentially useful as a temporal feature.
- day_of_week may provide weaker additional information and should be evaluated empirically rather than assumed to be important.

In [39]:
df["route"] = df["pickup"] + " -> " + df["delivery"]

In [40]:
print("Unique pickup cities:", df["pickup"].nunique())
print("Unique delivery cities:", df["delivery"].nunique())
print("Unique routes:", df["route"].nunique())

Unique pickup cities: 64
Unique delivery cities: 64
Unique routes: 4014


In [41]:
route_counts = df["route"].value_counts()

print(route_counts.describe())
print(route_counts.head(10))

count    4014.000000
mean       11.958146
std         7.165047
min         1.000000
25%         6.000000
50%        10.000000
75%        16.000000
max        39.000000
Name: count, dtype: float64
route
Columbia -> Oklahoma City     39
Phoenix -> Shreveport         39
Lexington -> Atlanta          39
Fort Wayne -> Philadelphia    38
Shreveport -> Hartford        37
Lexington -> Cincinnati       37
Fort Wayne -> Hartford        37
Richmond -> Phoenix           37
Richmond -> Oklahoma City     37
Lexington -> Kansas City      37
Name: count, dtype: int64


In [42]:
df.groupby("route")["posted_rate"].agg(
    ["mean", "median", "std", "count"]
).sort_values("count", ascending=False).head(20)

,mean,median,std,count
route,,,,
Columbia -> Oklahoma City,2330.734615,2327.430,172.229178,39
Phoenix -> Shreveport,3507.618462,3554.430,242.204457,39
Lexington -> Atlanta,560.106154,553.630,50.720487,39
Fort Wayne -> Philadelphia,1681.032632,1708.835,256.611701,38
Shreveport -> Hartford,2867.768108,2812.520,194.815806,37
Fort Wayne -> Hartford,1938.657297,1725.990,864.431736,37
Richmond -> Oklahoma City,3093.435946,3126.490,451.844903,37
Richmond -> Phoenix,5190.194595,5010.880,1301.128774,37
Lexington -> Kansas City,1042.474595,1029.800,67.641548,37


In [43]:
df.groupby("route")["rate_per_mile"].agg(
    ["mean", "median", "std", "count"]
).sort_values("count", ascending=False).head(20)

,mean,median,std,count
route,,,,
Columbia -> Oklahoma City,2.196130,2.197595,0.162411,39
Phoenix -> Shreveport,2.102597,2.125935,0.148100,39
Lexington -> Atlanta,2.679787,2.690923,0.192834,39
Fort Wayne -> Philadelphia,2.124232,2.158063,0.321139,38
Shreveport -> Hartford,2.001011,1.976371,0.133721,37
Fort Wayne -> Hartford,2.324156,2.087302,1.011570,37
Richmond -> Oklahoma City,2.026216,2.013583,0.297998,37
Richmond -> Phoenix,1.920692,1.868472,0.495377,37
Lexington -> Kansas City,2.327837,2.308000,0.144175,37


In [44]:
route_distance_stats = df.groupby("route")["distance"].agg(
    ["mean", "std", "min", "max", "count"]
)

route_distance_stats.sort_values("count", ascending=False).head(20)

,mean,std,min,max,count
route,,,,,
Columbia -> Oklahoma City,1061.415385,16.484930,1028.6,1102.1,39
Phoenix -> Shreveport,1668.494872,20.081806,1632.0,1711.0,39
Lexington -> Atlanta,208.943590,9.923094,184.9,228.4,39
Fort Wayne -> Philadelphia,791.602632,14.048728,767.8,813.7,38
Shreveport -> Hartford,1433.270270,24.659975,1385.6,1479.6,37
Fort Wayne -> Hartford,832.602703,17.215957,784.9,876.2,37
Richmond -> Oklahoma City,1526.840541,22.063399,1486.3,1570.9,37
Richmond -> Phoenix,2705.283784,41.437292,2617.1,2805.7,37
Lexington -> Kansas City,447.886486,10.791513,426.9,469.1,37


In [45]:
train_candidate = df[df["date"] < "2025-10-01"]
val_candidate = df[df["date"] >= "2025-10-01"]

print("Training rows:", len(train_candidate))
print("Validation rows:", len(val_candidate))

print("Training date range:",
      train_candidate["date"].min(),
      train_candidate["date"].max())

print("Validation date range:",
      val_candidate["date"].min(),
      val_candidate["date"].max())

Training rows: 43147
Validation rows: 4853
Training date range: 2025-01-01 00:00:00 2025-09-30 00:00:00
Validation date range: 2025-10-01 00:00:00 2025-10-31 00:00:00


In [46]:
print("Validation percentage:",
      len(val_candidate) / len(df) * 100)

Validation percentage: 10.110416666666666


### A chronological validation split was selected rather than a random split. Rows from January–September 2025 were used for training, while October 2025 was held out for validation. This produced 43,147 training rows and 4,853 validation rows, with validation representing approximately 10.1% of the dataset. The split was chosen to better simulate the real-world task of predicting future freight rates from historical data.

In [47]:
categorical_cols = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

for col in categorical_cols:
    train_values = set(train_candidate[col].dropna().unique())
    val_values = set(val_candidate[col].dropna().unique())

    unseen = val_values - train_values

    print(f"\n{col}")
    print("Train unique:", len(train_values))
    print("Validation unique:", len(val_values))
    print("Unseen in validation:", len(unseen))

    if len(unseen) > 0:
        print("Examples:", list(unseen)[:10])


pickup
Train unique: 64
Validation unique: 64
Unseen in validation: 0

delivery
Train unique: 64
Validation unique: 64
Unseen in validation: 0

equipment
Train unique: 3
Validation unique: 3
Unseen in validation: 0

route
Train unique: 4008
Validation unique: 2598
Unseen in validation: 6
Examples: ['Albuquerque -> Birmingham', 'Savannah -> Indianapolis', 'Birmingham -> Washington', 'Baltimore -> Las Vegas', 'El Paso -> Las Vegas', 'Washington -> Philadelphia']


All pickup cities, delivery cities, and equipment categories present in the October validation period were also observed during training. Of 2,598 unique routes appearing in October, only six were unseen during January–September (~0.23% of unique validation routes). This indicates strong route coverage while still requiring the model to handle a small number of unseen origin-destination combinations.

In [48]:
train_routes = set(train_candidate["route"].unique())
val_routes = set(val_candidate["route"].unique())

unseen_routes = val_routes - train_routes

unseen_route_rows = val_candidate[
    val_candidate["route"].isin(unseen_routes)
]

print("Unseen unique routes:", len(unseen_routes))
print("Validation rows using unseen routes:", len(unseen_route_rows))

unseen_route_rows[
    ["route", "distance", "equipment", "posted_rate"]
]

Unseen unique routes: 6
Validation rows using unseen routes: 8


,route,distance,equipment,posted_rate
43421,Savannah -> Indianapolis,588.5,Dry Van,1251.72
45248,Washington -> Philadelphia,279.6,Dry Van,672.87
45429,Albuquerque -> Birmingham,1463.8,Dry Van,2985.02
45641,Washington -> Philadelphia,274.4,Dry Van,648.83
45805,El Paso -> Las Vegas,344.7,Dry Van,859.09
46803,El Paso -> Las Vegas,341.2,Dry Van,786.95
46911,Baltimore -> Las Vegas,2803.2,Dry Van,5124.21
47284,Birmingham -> Washington,772.6,Dry Van,1658.47


Of the 4,853 October validation rows, only 8 belonged to origin-destination routes not observed in the January–September training period. Pickup cities, delivery cities, and equipment types had no unseen categories. Therefore, category coverage between training and validation is very strong, and route can remain a candidate feature as long as the modeling approach handles unseen categories safely.

In [49]:
validation_df = pd.read_csv("validation.csv")

print("Shape:", validation_df.shape)
print("Columns:", validation_df.columns)

validation_df.head()

Shape: (12000, 13)
Columns: Index(['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon',
       'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight',
       'date', 'market_index', 'quote_signal'],
      dtype='object')


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [50]:
print(validation_df.dtypes)
print(validation_df.isnull().sum())

load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
dtype: object
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          165
date              0
market_index    249
quote_signal      0
dtype: int64


In [51]:
validation_df["date"] = pd.to_datetime(validation_df["date"])

print("Earliest validation date:", validation_df["date"].min())
print("Latest validation date:", validation_df["date"].max())
print("Unique validation dates:", validation_df["date"].nunique())

Earliest validation date: 2025-11-01 00:00:00
Latest validation date: 2025-12-31 00:00:00
Unique validation dates: 61


## makig a copy , splitting it and start training 

In [52]:
model_df = df.copy()

In [53]:
model_df["month"] = model_df["date"].dt.month
model_df["day_of_week_num"] = model_df["date"].dt.dayofweek

In [54]:
model_df["route"] = (model_df["pickup"] + " -> " + model_df["delivery"])

In [55]:
train_model = model_df[model_df["date"] < "2025-10-01"].copy()
val_model = model_df[model_df["date"] >= "2025-10-01"].copy()

print(train_model.shape)
print(val_model.shape)

(43147, 19)
(4853, 19)


In [56]:
categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

numeric_features = [
    "distance",
    "weight",
    "month",
    "day_of_week_num"
]

target = "posted_rate"

In [57]:
X_train = train_model[categorical_features + numeric_features]
y_train = train_model[target]

X_val = val_model[categorical_features + numeric_features]
y_val = val_model[target]

In [58]:
print(X_train.shape)
print(X_val.shape)

X_train.head()

(43147, 8)
(4853, 8)


,pickup,delivery,equipment,route,distance,weight,month,day_of_week_num
0,Richmond,Baltimore,Dry Van,Richmond -> Baltimore,274.3,30658.0,1,2
1,Richmond,Philadelphia,Reefer,Richmond -> Philadelphia,280.5,17555.0,1,2
2,Philadelphia,Green Bay,Dry Van,Philadelphia -> Green Bay,967.8,31721.0,1,2
3,Hartford,Atlanta,Dry Van,Hartford -> Atlanta,965.4,32333.0,1,2
4,Dallas,Nashville,Reefer,Dallas -> Nashville,541.9,35183.0,1,2


In [59]:
from sklearn.pipeline import Pipeline

In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [61]:
from sklearn.linear_model import Ridge

baseline_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)

In [62]:
baseline_model.fit(X_train, y_train)

c:\Users\User\Downloads\.conda\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['pickup', 'delivery',
                                                   'equipment', 'route'])])),
                ('model', Ridge())])

In [63]:
baseline_predictions = baseline_model.predict(X_val)

In [64]:
print(baseline_predictions[:10])

[4450.14125831 2724.43076533 2488.45828452 1495.81204261 3269.25511002
 1825.7756281   803.11164324 1451.58816284 2160.49377643 2119.99274208]


In [65]:
comparison = pd.DataFrame({
    "actual_rate": y_val.values,
    "predicted_rate": baseline_predictions
})

comparison.head(10)

,actual_rate,predicted_rate
0,4001.45,4450.141258
1,2466.25,2724.430765
2,2526.28,2488.458285
3,1518.22,1495.812043
4,2999.76,3269.255110
5,1704.26,1825.775628
6,605.83,803.111643
7,1176.88,1451.588163
8,1900.80,2160.493776
9,1914.59,2119.992742


In [66]:
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

mae = mean_absolute_error(y_val, baseline_predictions)

rmse = np.sqrt(
    mean_squared_error(y_val, baseline_predictions)
)

r2 = r2_score(y_val, baseline_predictions)

mape = (
    mean_absolute_percentage_error(
        y_val,
        baseline_predictions
    ) * 100
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)
print("MAPE:", mape, "%")

MAE: 234.36388128533468
RMSE: 670.5248844748713
R²: 0.8075726533361147
MAPE: 15.20034060302555 %


In [67]:
comparison["absolute_error"] = (
    comparison["actual_rate"] -
    comparison["predicted_rate"]
).abs()

comparison.describe()

,actual_rate,predicted_rate,absolute_error
count,4853.000000,4853.000000,4853.000000
mean,2379.051374,2475.819079,234.363881
std,1528.714038,1352.692832,628.298126
min,104.380000,410.538509,0.289773
25%,1231.130000,1388.685686,99.950108
50%,2035.900000,2132.372090,169.018120
75%,3305.450000,3390.876828,233.363870
max,19110.300000,6752.811812,13718.278404


In [68]:
error_analysis = val_model[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "posted_rate"
    ]
].reset_index(drop=True)

error_analysis["predicted_rate"] = baseline_predictions

error_analysis["absolute_error"] = (
    error_analysis["posted_rate"] -
    error_analysis["predicted_rate"]
).abs()

error_analysis.nlargest(
    20,
    "absolute_error"
)

,pickup,delivery,distance,equipment,weight,date,posted_rate,predicted_rate,absolute_error
3066,Columbia,Tucson,2023.4,Flatbed,34710.0,2025-10-20,17893.17,4174.891596,13718.278404
392,Milwaukee,Bakersfield,1893.0,Dry Van,30760.0,2025-10-03,17514.11,3878.905514,13635.204486
4151,Boston,Bakersfield,2979.1,Reefer,30585.0,2025-10-27,19110.30,5919.566065,13190.733935
3369,Toledo,Albuquerque,1684.3,Reefer,31536.0,2025-10-22,14784.38,3525.940216,11258.439784
3219,Bakersfield,Columbia,2251.9,Dry Van,40729.0,2025-10-21,14403.14,4623.503247,9779.636753
155,Atlanta,Phoenix,2083.1,Reefer,40335.0,2025-10-01,13894.55,4328.788108,9565.761892
4389,Los Angeles,Richmond,2782.4,Dry Van,25730.0,2025-10-29,15003.55,5492.207986,9511.342014
1120,Albany,Albuquerque,2492.6,Dry Van,25513.0,2025-10-08,14425.42,4954.599325,9470.820675
237,Lexington,Lubbock,1187.0,Dry Van,25359.0,2025-10-02,11696.44,2535.691504,9160.748496
4439,Corpus Christi,Las Vegas,1383.3,Flatbed,18645.0,2025-10-29,11596.28,2867.751863,8728.528137


## important note 
Baseline Ridge regression achieved MAE ≈ $234, RMSE ≈ $671, R² ≈ 0.808, and MAPE ≈ 15.2%. Error analysis showed that most predictions were relatively accurate, but the largest errors occurred on rare, unusually high-priced loads. Ridge consistently underpredicted these extreme rates, indicating that a linear model may not capture nonlinear interactions between route, distance, equipment, and temporal factors.

In [69]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [70]:
from catboost import CatBoostRegressor

In [71]:
catboost_model = CatBoostRegressor(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

In [72]:
categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

In [73]:
catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100
)

0:	learn: 1423.7659853	test: 1469.9523113	best: 1469.9523113 (0)	total: 118ms	remaining: 1m 34s
100:	learn: 584.5180778	test: 650.0194567	best: 650.0194567 (100)	total: 3.57s	remaining: 24.7s
200:	learn: 577.4317945	test: 648.0984024	best: 648.0984024 (200)	total: 6.43s	remaining: 19.2s
300:	learn: 572.1903560	test: 648.0542548	best: 647.8407658 (237)	total: 9.26s	remaining: 15.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 647.8407658
bestIteration = 237

Shrink model to first 238 iterations.


CatBoostRegressor(depth=8, iterations=800, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [74]:
catboost_predictions = catboost_model.predict(X_val)

In [75]:
cat_mae = mean_absolute_error(
    y_val,
    catboost_predictions
)

cat_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        catboost_predictions
    )
)

cat_r2 = r2_score(
    y_val,
    catboost_predictions
)

cat_mape = (
    mean_absolute_percentage_error(
        y_val,
        catboost_predictions
    ) * 100
)

print("CatBoost MAE:", cat_mae)
print("CatBoost RMSE:", cat_rmse)
print("CatBoost R²:", cat_r2)
print("CatBoost MAPE:", cat_mape, "%")

CatBoost MAE: 122.05859425842891
CatBoost RMSE: 647.8407703786747
CatBoost R²: 0.8203722031563
CatBoost MAPE: 6.3134052769114755 %


In [76]:
cat_error_analysis = val_model[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date",
        "posted_rate"
    ]
].reset_index(drop=True)

cat_error_analysis["predicted_rate"] = catboost_predictions

cat_error_analysis["absolute_error"] = (
    cat_error_analysis["posted_rate"]
    - cat_error_analysis["predicted_rate"]
).abs()

cat_error_analysis.nlargest(
    20,
    "absolute_error"
)

,pickup,delivery,distance,equipment,weight,date,posted_rate,predicted_rate,absolute_error
392,Milwaukee,Bakersfield,1893.0,Dry Van,30760.0,2025-10-03,17514.11,3711.775983,13802.334017
3066,Columbia,Tucson,2023.4,Flatbed,34710.0,2025-10-20,17893.17,4264.151014,13629.018986
4151,Boston,Bakersfield,2979.1,Reefer,30585.0,2025-10-27,19110.30,5937.962808,13172.337192
3369,Toledo,Albuquerque,1684.3,Reefer,31536.0,2025-10-22,14784.38,3731.306939,11053.073061
3219,Bakersfield,Columbia,2251.9,Dry Van,40729.0,2025-10-21,14403.14,4351.398287,10051.741713
4389,Los Angeles,Richmond,2782.4,Dry Van,25730.0,2025-10-29,15003.55,4997.770908,10005.779092
1120,Albany,Albuquerque,2492.6,Dry Van,25513.0,2025-10-08,14425.42,4557.532906,9867.887094
155,Atlanta,Phoenix,2083.1,Reefer,40335.0,2025-10-01,13894.55,4486.008463,9408.541537
237,Lexington,Lubbock,1187.0,Dry Van,25359.0,2025-10-02,11696.44,2438.399011,9258.040989
4439,Corpus Christi,Las Vegas,1383.3,Flatbed,18645.0,2025-10-29,11596.28,2918.219758,8678.060242


In [77]:
rich_categorical_features = [
    "pickup",
    "delivery",
    "equipment",
    "route"
]

rich_numeric_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "month",
    "day_of_week_num"
]

rich_features = (
    rich_categorical_features +
    rich_numeric_features
)

In [78]:
X_train_rich = train_model[rich_features]
X_val_rich = val_model[rich_features]

In [79]:
rich_catboost = CatBoostRegressor(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

rich_catboost.fit(
    X_train_rich,
    y_train,
    cat_features=rich_categorical_features,
    eval_set=(X_val_rich, y_val),
    early_stopping_rounds=100,
    verbose=100
)

0:	learn: 1423.2229017	test: 1469.5122765	best: 1469.5122765 (0)	total: 40.5ms	remaining: 32.4s
100:	learn: 580.7576225	test: 648.7947587	best: 648.7936923 (99)	total: 3.76s	remaining: 26s
200:	learn: 572.3126274	test: 647.2721017	best: 647.2721017 (200)	total: 6.73s	remaining: 20.1s
300:	learn: 565.6169020	test: 649.4353660	best: 646.9480175 (223)	total: 9.8s	remaining: 16.3s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 646.9480175
bestIteration = 223

Shrink model to first 224 iterations.


CatBoostRegressor(depth=8, iterations=800, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [80]:
rich_predictions = rich_catboost.predict(X_val_rich)

In [81]:
cat_mae = mean_absolute_error(
    y_val,
    rich_predictions
)

cat_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        rich_predictions
    )
)

cat_r2 = r2_score(
    y_val,
    rich_predictions
)

cat_mape = (
    mean_absolute_percentage_error(
        y_val,
        rich_predictions
    ) * 100
)

print("CatBoost MAE:", cat_mae)
print("CatBoost RMSE:", cat_rmse)
print("CatBoost R²:", cat_r2)
print("CatBoost MAPE:", cat_mape, "%")

CatBoost MAE: 117.98312137082948
CatBoost RMSE: 646.9480221354925
CatBoost R²: 0.8208669293753463
CatBoost MAPE: 5.907307998693505 %


In [82]:
backtest_train = model_df[
    model_df["date"] < "2025-09-01"
].copy()

backtest_val = model_df[
    model_df["date"] >= "2025-09-01"
].copy()

print("Training rows:", len(backtest_train))
print("Validation rows:", len(backtest_val))

print(
    "Train:",
    backtest_train["date"].min(),
    backtest_train["date"].max()
)

print(
    "Validation:",
    backtest_val["date"].min(),
    backtest_val["date"].max()
)

Training rows: 38477
Validation rows: 9523
Train: 2025-01-01 00:00:00 2025-08-31 00:00:00
Validation: 2025-09-01 00:00:00 2025-10-31 00:00:00


In [83]:
X_backtest_train = backtest_train[
    categorical_features + numeric_features
]
y_backtest_train = backtest_train["posted_rate"]

X_backtest_val = backtest_val[
    categorical_features + numeric_features
]
y_backtest_val = backtest_val["posted_rate"]

In [84]:
backtest_common_model = CatBoostRegressor(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

backtest_common_model.fit(
    X_backtest_train,
    y_backtest_train,
    cat_features=categorical_features,
    eval_set=(X_backtest_val, y_backtest_val),
    early_stopping_rounds=100,
    verbose=100
)

backtest_common_predictions = backtest_common_model.predict(
    X_backtest_val
)

0:	learn: 1417.6478761	test: 1467.2774127	best: 1467.2774127 (0)	total: 35.6ms	remaining: 28.4s
100:	learn: 580.6557344	test: 633.1755151	best: 633.1755151 (100)	total: 3.58s	remaining: 24.8s
200:	learn: 571.9625547	test: 632.8604105	best: 632.6542329 (122)	total: 6.88s	remaining: 20.5s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 632.6542329
bestIteration = 122

Shrink model to first 123 iterations.


In [85]:
def evaluate_regression(y_true, predictions):
    mae = mean_absolute_error(y_true, predictions)
    rmse = np.sqrt(mean_squared_error(y_true, predictions))
    r2 = r2_score(y_true, predictions)
    mape = mean_absolute_percentage_error(
        y_true,
        predictions
    ) * 100

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R²:", r2)
    print("MAPE:", mape, "%")

In [86]:
evaluate_regression(
    y_backtest_val,
    backtest_common_predictions
)

MAE: 122.60194017069651
RMSE: 632.6542375549889
R²: 0.8281324183295036
MAPE: 6.1075880760788515 %


In [87]:
X_backtest_train_rich = backtest_train[rich_features]
X_backtest_val_rich = backtest_val[rich_features]

In [88]:
backtest_rich_model = CatBoostRegressor(
    iterations=800,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

backtest_rich_model.fit(
    X_backtest_train_rich,
    y_backtest_train,
    cat_features=categorical_features,
    eval_set=(X_backtest_val_rich, y_backtest_val),
    early_stopping_rounds=100,
    verbose=100
)

backtest_rich_predictions = backtest_rich_model.predict(
    X_backtest_val_rich
)

0:	learn: 1418.9820813	test: 1468.1284842	best: 1468.1284842 (0)	total: 33.9ms	remaining: 27.1s
100:	learn: 574.9354913	test: 633.7339477	best: 633.7339477 (100)	total: 3.67s	remaining: 25.4s
200:	learn: 564.7646719	test: 632.2109388	best: 632.1588415 (192)	total: 7.3s	remaining: 21.7s
300:	learn: 559.4698772	test: 632.2371981	best: 632.1080973 (207)	total: 11.5s	remaining: 19s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 632.1080973
bestIteration = 207

Shrink model to first 208 iterations.


In [89]:
evaluate_regression(
    y_backtest_val,
    backtest_rich_predictions
)

MAE: 108.61171687315117
RMSE: 632.1081018815931
R²: 0.8284290179521282
MAPE: 5.173961329535666 %


A second chronological backtest was performed using January–August for training and September–October for validation, approximating the two-month horizon of the final November–December prediction set. Both CatBoost variants remained stable. The common-feature model achieved MAE $122.60, RMSE $632.65, R² 0.8281, and MAPE 6.11%. The rich-feature model improved MAE to $108.61 and MAPE to 5.17%, while RMSE and R² remained similar. This confirmed that the additional market/geographic features provide incremental predictive value without destabilizing future-period

## supmittion ready section

In [90]:
dev_df = pd.read_csv("train-test.csv")
final_df = pd.read_csv("validation.csv")
december_df = pd.read_csv("december-chart-inputs.csv")

In [91]:
dev_df["date"] = pd.to_datetime(dev_df["date"])
final_df["date"] = pd.to_datetime(final_df["date"])
december_df["date"] = pd.to_datetime(december_df["date"])

In [92]:
dev_df["weight"] = dev_df["weight"].abs()
final_df["weight"] = final_df["weight"].abs()
december_df["weight"] = december_df["weight"].abs()

In [93]:
weight_median = dev_df["weight"].median()

print("Training weight median:", weight_median)

Training weight median: 31496.0


In [94]:
dev_df["weight"] = dev_df["weight"].fillna(weight_median)
final_df["weight"] = final_df["weight"].fillna(weight_median)
december_df["weight"] = december_df["weight"].fillna(weight_median)

In [95]:
dev_daily_market = (dev_df.groupby("date")["market_index"].transform("median"))

dev_df["market_index"] = (dev_df["market_index"].fillna(dev_daily_market))

In [96]:
final_daily_market = (final_df.groupby("date")["market_index"].transform("median"))

final_df["market_index"] = (final_df["market_index"].fillna(final_daily_market))

In [97]:
market_fallback = dev_df["market_index"].median()

dev_df["market_index"] = (dev_df["market_index"].fillna(market_fallback))

final_df["market_index"] = (final_df["market_index"].fillna(market_fallback))

In [98]:
def add_features(frame):
    frame = frame.copy()

    frame["month"] = frame["date"].dt.month
    frame["day_of_week_num"] = frame["date"].dt.dayofweek

    frame["route"] = (frame["pickup"]+ " -> "+ frame["delivery"])

    return frame

In [99]:
dev_df = add_features(dev_df)
final_df = add_features(final_df)
december_df = add_features(december_df)

In [100]:
print("Development missing:")
print(dev_df.isnull().sum())

print("\nFinal validation missing:")
print(final_df.isnull().sum())

print("\nDecember missing:")
print(december_df.isnull().sum())

Development missing:
load_id            0
pickup             0
delivery           0
pickup_lat         0
pickup_lon         0
delivery_lat       0
delivery_lon       0
distance           0
equipment          0
weight             0
date               0
market_index       0
quote_signal       0
posted_rate        0
month              0
day_of_week_num    0
route              0
dtype: int64

Final validation missing:
load_id            0
pickup             0
delivery           0
pickup_lat         0
pickup_lon         0
delivery_lat       0
delivery_lon       0
distance           0
equipment          0
weight             0
date               0
market_index       0
quote_signal       0
month              0
day_of_week_num    0
route              0
dtype: int64

December missing:
pickup              0
delivery            0
distance            0
equipment           0
weight              0
date                0
predicted_rate     31
month               0
day_of_week_num     0
route           

In [101]:
print(dev_df.shape)
print(final_df.shape)
print(december_df.shape)

(48000, 17)
(12000, 16)
(31, 10)


In [102]:
final_rich_model = CatBoostRegressor(
    iterations=220,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

X_dev_rich = dev_df[rich_features]
y_dev = dev_df["posted_rate"]

X_final_rich = final_df[rich_features]

final_rich_model.fit(
    X_dev_rich,
    y_dev,
    cat_features=rich_categorical_features
)

final_predictions = final_rich_model.predict(X_final_rich)

0:	learn: 1427.8960669	total: 68.3ms	remaining: 15s
100:	learn: 586.0444954	total: 7.99s	remaining: 9.42s
200:	learn: 576.4646731	total: 15.4s	remaining: 1.46s
219:	learn: 575.0813544	total: 16.6s	remaining: 0us


In [103]:
print("Predictions:", len(final_predictions))
print("Min:", final_predictions.min())
print("Max:", final_predictions.max())
print("Mean:", final_predictions.mean())
print("Non-positive:", (final_predictions <= 0).sum())

Predictions: 12000
Min: 257.7218503579488
Max: 6844.111807246676
Mean: 2393.234663572118
Non-positive: 0


In [104]:
template = pd.read_csv("validation-predictions-template.csv")

prediction_map = dict(
    zip(final_df["load_id"], final_predictions)
)

template["predicted_rate"] = (
    template["load_id"].map(prediction_map)
)

print(template.isnull().sum())
print(template.shape)
template.head()

load_id           0
predicted_rate    0
dtype: int64
(12000, 2)


,load_id,predicted_rate
0,TE-000001,861.250522
1,TE-000002,5051.793995
2,TE-000003,5313.958336
3,TE-000004,4087.334492
4,TE-000005,1838.183325


In [105]:
template.to_csv("validation_predictions.csv",index=False)

In [106]:
final_common_model = CatBoostRegressor(
    iterations=230,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

X_dev_common = dev_df[
    categorical_features + numeric_features
]

X_december = december_df[
    categorical_features + numeric_features
]

final_common_model.fit(
    X_dev_common,
    y_dev,
    cat_features=categorical_features
)

december_predictions = final_common_model.predict(
    X_december
)

0:	learn: 1427.5348798	total: 79.2ms	remaining: 18.1s
100:	learn: 590.3742879	total: 7.33s	remaining: 9.36s
200:	learn: 582.3289711	total: 13.9s	remaining: 2s
229:	learn: 580.3972052	total: 15s	remaining: 0us


In [107]:
print("December predictions:", len(december_predictions))
print("Min:", december_predictions.min())
print("Max:", december_predictions.max())
print("Non-positive:", (december_predictions <= 0).sum())

December predictions: 31
Min: 868.854268790788
Max: 898.0227613243719
Non-positive: 0


In [108]:
december_submission = december_df[
    [
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "weight",
        "date"
    ]
].copy()

december_submission["predicted_rate"] = (
    december_predictions
)

december_submission["date"] = (
    december_submission["date"]
    .dt.strftime("%Y-%m-%d")
)

december_submission.to_csv(
    "december_chart_inputs_completed.csv",
    index=False
)

december_submission.head()

,pickup,delivery,distance,equipment,weight,date,predicted_rate
0,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-01,868.854269
1,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-02,881.816759
2,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-03,898.022761
3,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-04,896.745990
4,Lexington,Fort Wayne,360,Dry Van,32000,2025-12-05,890.451177
